In [1]:
import pandas as pd

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\mohma\.cache\kagglehub\datasets\jp797498e\twitter-entity-sentiment-analysis\versions\2


In [3]:
col_name=["ID","Entity","Labels","Text"]
data =pd.read_csv(r"C:\Users\mohma\.cache\kagglehub\datasets\jp797498e\twitter-entity-sentiment-analysis\versions\2\twitter_training.csv",names=col_name)
data_vil=pd.read_csv(r"C:\Users\mohma\.cache\kagglehub\datasets\jp797498e\twitter-entity-sentiment-analysis\versions\2\twitter_validation.csv")

In [4]:
data

,ID,Entity,Labels,Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
...,...,...,...,...
74677,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74678,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74679,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74680,9200,Nvidia,Positive,Just realized between the windows partition of...


In [5]:
data

,ID,Entity,Labels,Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
...,...,...,...,...
74677,9200,Nvidia,Positive,Just realized that the Windows partition of my...
74678,9200,Nvidia,Positive,Just realized that my Mac window partition is ...
74679,9200,Nvidia,Positive,Just realized the windows partition of my Mac ...
74680,9200,Nvidia,Positive,Just realized between the windows partition of...


In [6]:
data["Labels"].value_counts()

Labels
Negative      22542
Positive      20832
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64

In [7]:
data = data[data['Labels']!= 'Irrelevant']

In [8]:
data['Labels'].value_counts()

Labels
Negative    22542
Positive    20832
Neutral     18318
Name: count, dtype: int64

In [9]:
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder()
data['Labels'] = encoder.fit_transform(data['Labels'])

C:\Users\mohma\AppData\Local\Temp\ipykernel_18528\1969877191.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Labels'] = encoder.fit_transform(data['Labels'])


In [10]:
data

,ID,Entity,Labels,Text
0,2401,Borderlands,2,im getting on borderlands and i will murder yo...
1,2401,Borderlands,2,I am coming to the borders and I will kill you...
2,2401,Borderlands,2,im getting on borderlands and i will kill you ...
3,2401,Borderlands,2,im coming on borderlands and i will murder you...
4,2401,Borderlands,2,im getting on borderlands 2 and i will murder ...
...,...,...,...,...
74677,9200,Nvidia,2,Just realized that the Windows partition of my...
74678,9200,Nvidia,2,Just realized that my Mac window partition is ...
74679,9200,Nvidia,2,Just realized the windows partition of my Mac ...
74680,9200,Nvidia,2,Just realized between the windows partition of...


In [11]:
data['Labels'].value_counts()  #negative = 0, positive = 2 , Neutral = 1

Labels
0    22542
2    20832
1    18318
Name: count, dtype: int64

In [12]:
data.head()

,ID,Entity,Labels,Text
0,2401,Borderlands,2,im getting on borderlands and i will murder yo...
1,2401,Borderlands,2,I am coming to the borders and I will kill you...
2,2401,Borderlands,2,im getting on borderlands and i will kill you ...
3,2401,Borderlands,2,im coming on borderlands and i will murder you...
4,2401,Borderlands,2,im getting on borderlands 2 and i will murder ...


In [13]:
data = data[["Text","Labels"]]

In [14]:
data.isna().sum()

Text      571
Labels      0
dtype: int64

In [15]:
data["Text"].size

61692

In [16]:
import re

def clean_text(text):
    text = str(text).lower()  # Hello World! → hello world!
    text = re.sub(r"http\S+", "", text) # http://example.com → ""
    text = re.sub(r"@\w+", "", text) # @user123 → ""
    text = re.sub(r"#", "", text) # "#iphone" → "iphone"
    text = re.sub(r"[^a-zA-Z ]", " ", text) # hello!!! → hello
    text = re.sub(r"\s+", " ", text).strip() #"hello     world  " → "hello world"
    return text



data["clean_text"] = data["Text"].apply(clean_text)


C:\Users\mohma\AppData\Local\Temp\ipykernel_18528\3375403604.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["clean_text"] = data["Text"].apply(clean_text)


In [17]:
data["Labels"]

0        2
1        2
2        2
3        2
4        2
        ..
74677    2
74678    2
74679    2
74680    2
74681    2
Name: Labels, Length: 61692, dtype: int64

In [18]:
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split

# 1) Split
X_train, X_test, y_train, y_test = train_test_split(
    data["clean_text"], data["Labels"], test_size=0.2, random_state=42
)

# 2) Tokenizer fit ONLY on train
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

# 3) Transform train & test
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [19]:
data["clean_text"]

0        im getting on borderlands and i will murder yo...
1        i am coming to the borders and i will kill you...
2        im getting on borderlands and i will kill you all
3        im coming on borderlands and i will murder you...
4        im getting on borderlands and i will murder yo...
                               ...                        
74677    just realized that the windows partition of my...
74678    just realized that my mac window partition is ...
74679    just realized the windows partition of my mac ...
74680    just realized between the windows partition of...
74681    just like the windows partition of my mac is l...
Name: clean_text, Length: 61692, dtype: object

In [20]:
data

,Text,Labels,clean_text
0,im getting on borderlands and i will murder yo...,2,im getting on borderlands and i will murder yo...
1,I am coming to the borders and I will kill you...,2,i am coming to the borders and i will kill you...
2,im getting on borderlands and i will kill you ...,2,im getting on borderlands and i will kill you all
3,im coming on borderlands and i will murder you...,2,im coming on borderlands and i will murder you...
4,im getting on borderlands 2 and i will murder ...,2,im getting on borderlands and i will murder yo...
...,...,...,...
74677,Just realized that the Windows partition of my...,2,just realized that the windows partition of my...
74678,Just realized that my Mac window partition is ...,2,just realized that my mac window partition is ...
74679,Just realized the windows partition of my Mac ...,2,just realized the windows partition of my mac ...
74680,Just realized between the windows partition of...,2,just realized between the windows partition of...


In [21]:
def extract_hashtags(text):
    text = str(text)
    return re.findall(r"#(\w+)", text)   

def extract_mentions(text):
    text = str(text)
    return re.findall(r"@(\w+)", text) 

data["hashtags"] = data["Text"].apply(extract_hashtags)
data["mentions"] = data["Text"].apply(extract_mentions)


In [22]:
data.to_csv("data_new.csv",index=False)

In [23]:
from sklearn.model_selection import train_test_split

# 1) Split
X_train, X_test, y_train, y_test = train_test_split(
    data["clean_text"], data["Labels"], test_size=0.2, random_state=42,stratify=data["Labels"]
)

In [24]:
from tensorflow.keras.preprocessing.text import Tokenizer


tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)


In [25]:
X_test_seq

[[1028,
  67,
  193,
  58,
  3619,
  4363,
  104,
  103,
  223,
  2373,
  129,
  7405,
  4,
  17504,
  172,
  156,
  21,
  507,
  9,
  2979,
  4,
  1629,
  9,
  346,
  16,
  15,
  507,
  2077,
  5795,
  2,
  31,
  835,
  163,
  230,
  77,
  34,
  11,
  17,
  198],
 [601,
  1786,
  2,
  118,
  16,
  310,
  2327,
  30,
  27,
  1817,
  10,
  9048,
  26,
  2,
  107,
  103,
  11,
  10,
  49,
  494,
  4,
  418,
  7,
  53,
  3,
  23,
  15645,
  317,
  45,
  116,
  1142,
  1,
  3226,
  8,
  1032,
  3842,
  4,
  274,
  27,
  1543,
  33,
  28,
  1084],
 [1922,
  32,
  5,
  90,
  432,
  53,
  1828,
  378,
  524,
  448,
  907,
  121,
  245,
  17020,
  2734,
  1292,
  53,
  4957,
  393,
  33],
 [9125, 2879, 1036, 6, 3338, 58, 5, 90, 1927, 1261, 1638, 1286, 1261],
 [2, 1495, 18165, 1132, 9, 9151],
 [332, 56],
 [1252,
  47,
  12,
  179,
  258,
  61,
  49,
  1,
  337,
  1321,
  50,
  15,
  1,
  447,
  6,
  254,
  5,
  83,
  1234,
  13,
  2792,
  59,
  61,
  129,
  511,
  6,
  14,
  77,
  5600,
  741,


In [26]:
data["hashtags"].value_counts()

hashtags
[]    61692
Name: count, dtype: int64

In [27]:
data.columns

Index(['Text', 'Labels', 'clean_text', 'hashtags', 'mentions'], dtype='object')

In [28]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

#maxlen=200

max_len = max(len(x) for x in X_train_seq)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [29]:
X_test_seq

[[1028,
  67,
  193,
  58,
  3619,
  4363,
  104,
  103,
  223,
  2373,
  129,
  7405,
  4,
  17504,
  172,
  156,
  21,
  507,
  9,
  2979,
  4,
  1629,
  9,
  346,
  16,
  15,
  507,
  2077,
  5795,
  2,
  31,
  835,
  163,
  230,
  77,
  34,
  11,
  17,
  198],
 [601,
  1786,
  2,
  118,
  16,
  310,
  2327,
  30,
  27,
  1817,
  10,
  9048,
  26,
  2,
  107,
  103,
  11,
  10,
  49,
  494,
  4,
  418,
  7,
  53,
  3,
  23,
  15645,
  317,
  45,
  116,
  1142,
  1,
  3226,
  8,
  1032,
  3842,
  4,
  274,
  27,
  1543,
  33,
  28,
  1084],
 [1922,
  32,
  5,
  90,
  432,
  53,
  1828,
  378,
  524,
  448,
  907,
  121,
  245,
  17020,
  2734,
  1292,
  53,
  4957,
  393,
  33],
 [9125, 2879, 1036, 6, 3338, 58, 5, 90, 1927, 1261, 1638, 1286, 1261],
 [2, 1495, 18165, 1132, 9, 9151],
 [332, 56],
 [1252,
  47,
  12,
  179,
  258,
  61,
  49,
  1,
  337,
  1321,
  50,
  15,
  1,
  447,
  6,
  254,
  5,
  83,
  1234,
  13,
  2792,
  59,
  61,
  129,
  511,
  6,
  14,
  77,
  5600,
  741,


In [30]:
df_train = pd.DataFrame(X_train_seq)
df_train.to_csv("X_train_data.csv", index=False)

df_test = pd.DataFrame(X_test_seq)
df_test.to_csv("X_test_data.csv", index=False)

df_test = pd.DataFrame(y_test)
df_test.to_csv("y_test_data.csv", index=False)
df_test = pd.DataFrame(y_train)
df_test.to_csv("y_train_data.csv", index=False)

In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *